# 💳 Credit Card Fraud — Anomaly Detection

**Framework:** CRISP-DM  
**Domain:** Fraud analytics  
**Methods:** Isolation Forest, Local Outlier Factor, One-Class SVM  
**Dataset:** Kaggle / ULB Credit Card Fraud Detection

The notebook automatically attempts to download the **full OpenML copy of the Kaggle benchmark**. If internet access is unavailable, it uses an explicitly labeled offline validation proxy so the pipeline remains runnable.

> Never report offline-fallback metrics as Kaggle benchmark performance.


## 1. Business Understanding

### Objective
Rank suspicious credit-card transactions for investigation without relying on a conventional supervised classifier.

### Why anomaly detection?
Fraud is a rare-event problem. The benchmark contains 284,807 transactions but only 492 fraud cases, so ordinary accuracy is misleading.

### Operational success criteria
A useful detector should:
- rank real fraud near the top,
- achieve strong **Average Precision / PR-AUC**,
- preserve high **recall**,
- keep the manual-review queue small,
- avoid excessive false alarms.


In [ ]:

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml, make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.metrics import (
    average_precision_score, roc_auc_score, precision_score,
    recall_score, f1_score, confusion_matrix, precision_recall_curve
)

SEED = 42

# Prefer the real benchmark.
try:
    data = fetch_openml(data_id=1597, as_frame=True, parser="auto")
    df = data.frame.copy()
    df["Class"] = pd.to_numeric(df["Class"], errors="coerce").astype(int)
    DATA_MODE = "FULL OPENML / KAGGLE BENCHMARK"
except Exception:
    # Offline fallback only so the notebook still runs without internet.
    # Do NOT present fallback metrics as Kaggle benchmark results.
    rng = np.random.default_rng(SEED)
    X_base, y = make_classification(
        n_samples=12000, n_features=28, n_informative=10, n_redundant=8,
        weights=[0.994,0.006], class_sep=2.2, flip_y=0.001, random_state=SEED
    )
    df = pd.DataFrame(X_base, columns=[f"V{i}" for i in range(1,29)])
    df.insert(0, "Time", rng.uniform(0,172800,len(df)))
    df["Amount"] = np.where(
        y==1,
        rng.lognormal(3.4,1.0,len(df)),
        rng.lognormal(3.0,.9,len(df))
    )
    df["Class"] = y.astype(int)
    DATA_MODE = "OFFLINE VALIDATION PROXY"

print("Mode:", DATA_MODE)
print("Shape:", df.shape)
display(df.head())


## 2. Data Understanding

The benchmark contains:
- `Time`
- `Amount`
- anonymized PCA features `V1`–`V28`
- `Class` where 1 = fraud and 0 = legitimate

We inspect shape, missing values, class imbalance, transaction amounts, and feature distributions.


In [ ]:
print(df.info())
print("\nMissing values:", int(df.isna().sum().sum()))
print("\nClass counts:")
display(df["Class"].value_counts().rename(index={0:"Legitimate",1:"Fraud"}).to_frame("count"))

fraud_rate = df["Class"].mean()
print(f"Fraud rate: {fraud_rate:.4%}")


In [ ]:
counts = df["Class"].value_counts().sort_index()
plt.figure(figsize=(8,5))
plt.bar(["Legitimate","Fraud"], counts.values)
plt.ylabel("Transactions")
plt.title("Class Imbalance")
plt.tight_layout()
plt.show()


## 3. Data Preparation

Anomaly models are fit without using fraud labels as predictors.

Steps:
1. separate `Class` from features;
2. robust-scale `Time` and `Amount`;
3. preserve PCA features;
4. use labels **only for final evaluation**;
5. fit detectors primarily on normal transactions.


In [ ]:
features = [c for c in df.columns if c != "Class"]
X = df[features].copy()
y = df["Class"].astype(int).to_numpy()

scaler = RobustScaler()
X[["Time","Amount"]] = scaler.fit_transform(X[["Time","Amount"]])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=.35, stratify=y, random_state=SEED
)

normal_train = X_train[y_train == 0]

print("Train:", X_train.shape)
print("Test :", X_test.shape)
print("Normal-only fitting rows:", len(normal_train))


## 4. Modeling — Popular Anomaly Detection Methods

### Isolation Forest
Randomly partitions the feature space; anomalies usually require fewer splits to isolate.

### Local Outlier Factor
Measures how isolated a point is relative to its local neighborhood density.

### One-Class SVM
Learns a boundary around normal behavior and flags points outside it.

These are standard unsupervised / semi-supervised anomaly-detection approaches.


In [ ]:
models = {
    "Isolation Forest": IsolationForest(
        n_estimators=250, contamination="auto",
        random_state=SEED, n_jobs=-1
    ),
    "Local Outlier Factor": LocalOutlierFactor(
        n_neighbors=35, novelty=True,
        contamination="auto", n_jobs=-1
    ),
    "One-Class SVM": OneClassSVM(
        kernel="rbf", gamma="scale", nu=0.01
    )
}

scores = {}

for name, model in models.items():
    fit_data = normal_train
    # Keep OCSVM practical on full benchmark data.
    if name == "One-Class SVM" and len(fit_data) > 10000:
        fit_data = fit_data.sample(10000, random_state=SEED)

    model.fit(fit_data)
    scores[name] = -model.decision_function(X_test)
    print(name, "done")


## 5. Evaluation

Because the class is extremely imbalanced, **accuracy is not a useful primary metric**.

We evaluate:
- Average Precision / PR-AUC
- ROC-AUC
- precision
- recall
- F1
- top-1% review queue capture

Thresholding is framed as an operations decision: investigate the highest-risk 1% of transactions.


In [ ]:
results = []
predictions = {}

for name, score in scores.items():
    threshold = np.quantile(score, .99)
    pred = (score >= threshold).astype(int)
    predictions[name] = pred

    results.append({
        "Model": name,
        "Average Precision": average_precision_score(y_test, score),
        "ROC-AUC": roc_auc_score(y_test, score),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "Flagged": pred.sum()
    })

results_df = pd.DataFrame(results).sort_values(
    ["Average Precision","Recall"], ascending=False
)
display(results_df)


In [ ]:
plt.figure(figsize=(9,6))
for name, score in scores.items():
    precision, recall, _ = precision_recall_curve(y_test, score)
    ap = average_precision_score(y_test, score)
    plt.plot(recall, precision, label=f"{name} (AP={ap:.3f})")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall Curves")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
best_name = results_df.iloc[0]["Model"]
best_score = scores[best_name]
best_pred = predictions[best_name]

k = max(1, int(len(y_test)*.01))
top_idx = np.argsort(best_score)[::-1][:k]

topk_precision = y_test[top_idx].mean()
topk_recall = y_test[top_idx].sum() / max(y_test.sum(),1)

print("Best model:", best_name)
print(f"Top-1% precision: {topk_precision:.2%}")
print(f"Top-1% recall:    {topk_recall:.2%}")

cm = confusion_matrix(y_test, best_pred)
display(pd.DataFrame(
    cm,
    index=["Actual Legit","Actual Fraud"],
    columns=["Pred Legit","Pred Fraud"]
))


## 6. Deployment — Fraud Operations

The deployment output is not simply a binary label. It is a **ranked investigation queue**.

### Operational workflow
1. score each incoming transaction;
2. rank by anomaly score;
3. review the highest-risk transactions first;
4. track false alarms and fraud capture;
5. adjust the review threshold to staffing capacity;
6. retrain when transaction behavior drifts.

### Monitoring KPIs
- fraud recall
- precision among reviewed transactions
- PR-AUC
- alert volume
- false-positive rate
- fraud captured in top-k alerts


# CRISP-DM Conclusion

**Business Understanding → Data Understanding → Data Preparation → Modeling → Evaluation → Deployment**

The key lesson is that anomaly detection is a **ranking and triage problem**. On extremely imbalanced fraud data, the practical question is not “how accurate is the model?” but “how much fraud can we capture within a realistic review budget?”
